In [2]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [3]:
data_clean = "goodreads_onestar_clean.csv"   # cleaned data, one-star reviews only
TEXT_COL = "review_text"   # name of the column with the review text
MIN_WORDS = 10             # skip reviews shorter than this many words
SEED = 42 # random seed for reproducibility

HIDE = ["alice", "wonderland", "carroll", "silverstein", "peter"] # common titles and authors to hide

In [4]:
one_star = pd.read_csv(data_clean) # turning into a DataFrame
one_star = one_star.dropna(subset=[TEXT_COL]) # dropping reviews with no text
one_star = one_star[one_star[TEXT_COL].str.split().str.len() >= MIN_WORDS] # splitting review text into list of words, counting them, and keeping only reviews with at least MIN_WORDS words

print(f"1-star reviews: {len(one_star)}") # printing the number of 1-star reviews after filtering

1-star reviews: 8801


### Flagging reviews about "appropriateness"

In [5]:
def compare_groups(texts_a, texts_b):  # scoring words and phrases that are distinctive to one group vs another
    texts_a = list(texts_a) # a positive score = more typical of group A, negative score = more typical of group B
    texts_b = list(texts_b)

    counter = CountVectorizer( # using CountVectorizer to count words and phrases in the two groups
        ngram_range=(1, 2), # defining upper and lower limit of words to be extracted
        min_df=15,    # words/phrases must appear in 15+ reviews
        token_pattern=r"(?u)\b\w[\w']*\b",   # keeps negations and other words with apostrophes like "don't" 
    )
    counts = counter.fit_transform(texts_a + texts_b) # makes table of counts of each word/phrase in each review
        
    a_review_number = len(texts_a) # how many reviews in group A and group B
    b_review_number = len(texts_b)
    count_a = np.asarray(counts[:a_review_number].sum(axis=0)).ravel() # selects group A reviews, sums the counts of each term across all reviews, and flattens into list of numbers
    count_b = np.asarray(counts[a_review_number:].sum(axis=0)).ravel() # selects group B reviews, sums the counts of each term across all reviews, and flattens into list of numbers

    baseline = count_a + count_b # how many times each term appears in all reviews (group A + group B)
    baseline_total = baseline.sum() # sums across all terms to get the total number of term appearances in all reviews

    odds_a = np.log((count_a + baseline) / (count_a.sum() + baseline_total - count_a - baseline)) # computes the log-odds of each term appearing in group A vs group B, with a baseline to avoid division by zero
    odds_b = np.log((count_b + baseline) / (count_b.sum() + baseline_total - count_b - baseline))
    difference = odds_a - odds_b

    # Turn the difference into a score that accounts for how much data there is
    uncertainty = np.sqrt(1 / (count_a + baseline) + 1 / (count_b + baseline)) # this calculates uncertainty for each term based on its counts in both groups and the baseline 
    score = difference / uncertainty # dividing to get a z-score-like measure

    results = pd.DataFrame({ # putting everything into a DataFrame 
        "term": counter.get_feature_names_out(),
        "score": score,
        "count_A": count_a,
        "count_B": count_b,
    })

        
    for name in HIDE: # hiding common titles and authors from the results
        results = results[~results["term"].str.contains(r"\b" + name + r"\b")]

    return results.sort_values("score", ascending=False).reset_index(drop=True) # sorting the results by score in descending order and resetting the index


In [6]:
# These rules only record which phrase caused a review to enter the candidate set.
# They are NOT hand-label categories and do not describe what is inappropriate.
SCREENING_RULE_PATTERNS = {
    "explicit_phrase": (
        r"\b(?:age[- ]?)?inappropriate\w*\b"
        r"|\b(?:not|isn't|wasn't|aren't|weren't)\s+(?:\w+\s+){0,2}(?:age[- ]?)?appropriate\b"
        r"|\bnot\s+suitable\b|\bunsuitable\b"
    ),
    "evaluative_term": r"\b(?:offensive|vulgar|crude|perverse|immoral|sinful)\b",
    "intensity_phrase": r"\btoo\s+(?:scary|violent|graphic|dark|mature|disturbing|sexual|adult|intense|creepy)\b",
    "age_fit_phrase": r"\btoo\s+(?:old|young|mature|advanced)\s+for\b",
    "audience_phrase": (
        r"\bnot\s+(?:for|meant\s+for|intended\s+for)\s+(?:\w+\s+){0,2}(?:kids|children|young)\b"
        r"|\bnot\s+(?:very\s+)?(?:child|kid|family)[- ]friendly\b"
    ),
}

# Use a non-capturing pattern for screening and named groups for match details.
TRIGGER_PATTERN = "|".join(f"(?:{pattern})" for pattern in SCREENING_RULE_PATTERNS.values())
LABELED_TRIGGER_PATTERN = "|".join(
    f"(?P<{name}>{pattern})" for name, pattern in SCREENING_RULE_PATTERNS.items()
)
WINDOW = 12

In [7]:
has_trigger = one_star[TEXT_COL].str.contains(TRIGGER_PATTERN, case=False, regex=True) # checking which reviews contain the trigger pattern, ignoring case and using regex
matched = one_star[has_trigger]
others = one_star[~has_trigger]
print(f"{len(matched):,} of {len(one_star):,} 1-star reviews contain an " # printing the number and percentage of 1-star reviews that contain an inappropriateness complaint
      f"inappropriateness complaint ({len(matched) / len(one_star):.1%})")

224 of 8,801 1-star reviews contain an inappropriateness complaint (2.5%)


In [8]:
def first_match_details(text, width=WINDOW):
    m = re.search(LABELED_TRIGGER_PATTERN, text, flags=re.I)
    before = " ".join(text[:m.start()].split()[-width:])
    after = " ".join(text[m.end():].split()[:width])
    return pd.Series({
        "matched_trigger": m.group(0),
        "screening_rule": m.lastgroup,
        "snippet": f"{before} [[{m.group(0)}]] {after}",
    })


def all_triggers(text):
    # Preserve order while removing duplicate matched phrases.
    matches = (m.group(0) for m in re.finditer(LABELED_TRIGGER_PATTERN, text, flags=re.I))
    return " | ".join(dict.fromkeys(matches))


matched = matched.copy()
matched[["matched_trigger", "screening_rule", "snippet"]] = matched[TEXT_COL].apply(first_match_details)
matched["all_triggers"] = matched[TEXT_COL].apply(all_triggers)

pd.set_option("display.max_colwidth", None) # letting the entire snippet show up in the output, instead of being truncated
print("\n25 random examples (read these first):")
print(matched[["book_id", "screening_rule", "snippet"]].sample(min(25, len(matched)), # printing 25 random examples of the matched reviews, showing the book_id and the snippet, and using a random seed for reproducibility
                                             random_state=SEED).to_string())


25 random examples (read these first):
       book_id    screening_rule                                                                                                                                                            snippet
105     237564   audience_phrase                             This book is a choking hazard and contains small parts. It is [[not for children]] under 3 years." That said, I read it to my 16 month
4087  16113648   evaluative_term                                                                                          Yup. Every bit as [[offensive]] as all the reviews and articles attested.
5639    389627   evaluative_term                    failures is because at an early age these days they're taught that [[crude]] humor is funny, that apathy and laziness is cute and that bullying
6647     65119  intensity_phrase                     I'm glad someone enjoys these books really. But the material is way [[too violent]] for young kids and yet the writing styl

In [9]:
distinct = compare_groups(matched[TEXT_COL], others[TEXT_COL]) # running the compare_groups function on the matched and others DataFrames to get the most distinctive words in the matched reviews
distinct = distinct[~distinct["term"].str.contains("appropriat|suitab|offen")] # removing the trigger words from the results, so that the most distinctive words are not just the trigger words themselves
distinct.to_csv("words_inappropriate_vs_other_1star.csv", index=False)
print("\nMost distinctive words in reviews that call something inappropriate:")
print(distinct.head(40).round(2).to_string())


Most distinctive words in reviews that call something inappropriate:
             term  score  count_A  count_B
5         old for   4.09       18        5
6         too old   3.39       18       15
7            rose   3.36       23       30
9            dark   2.98       45      159
10             oz   2.87       25       54
11            god   2.82       51      217
12  www goodreads   2.79       15       18
13  goodreads com   2.74       15       19
14       the dark   2.54       21       49
15      https www   2.45       14       22
16         my kid   2.45       17       34
17          bible   2.39       23       66
18          of oz   2.19       14       29
19          https   2.16       14       30
20           rain   2.04       13       29
21       children   2.02      158     1651
22          god's   1.97        9       14
23      preschool   1.97       11       22
24            www   1.92       19       68
25   for children   1.85       36      212
26           2014   1.83   

#### CSV containing all candidate reviews and blank HAND LABEL columns

`screening_rule` and `matched_trigger` only document why a review was retrieved. They must not be used as the final hand-label categories. The substantive category belongs in the blank `aspect` column after reading the full review. All reviews that match the screening rules are exported; the code does not impose a target sample size.

In [10]:
# Export every candidate rather than dropping reviews through random sampling.
# The original row index is retained as a stable ID when no review_id is available.
matched = matched.rename_axis("source_row_id").reset_index()
id_columns = [column for column in ["review_id", "source_row_id", "book_id"] if column in matched.columns]
screening_columns = ["matched_trigger", "screening_rule", "all_triggers", "snippet"]
to_code = matched[id_columns + [TEXT_COL] + screening_columns].copy()

# These are the only labeling fields. Screening metadata above is not a label.
to_code["is_inappropriate_complaint"] = ""
to_code["aspect"] = ""
to_code["notes"] = ""

to_code.to_csv("inappropriate_to_code.csv", index=False, encoding="utf-8-sig")
print("\nSaved inappropriate_to_code.csv with", len(to_code), "reviews to hand label")


Saved inappropriate_to_code.csv with 224 reviews to hand label


In [11]:
hand_labeled = pd.read_csv("inappropriate_to_code.csv", encoding="utf-8-sig")



### Looking at NOT-flagged reviews (reviews not about "appropriateness")

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print("\nWords most typical of the NOT-flagged reviews:")
print(distinct.sort_values("score").head(40).round(2).to_string(index=False)) # printing the 40 words with the lowest score, which are the most typical of the NOT-flagged reviews



Words most typical of the NOT-flagged reviews:
        term  score  count_A  count_B
           i  -1.12      989    22475
      boring  -0.84       18      759
    couldn't  -0.83       11      559
 interesting  -0.82       10      523
  half stars  -0.80        1      229
        baby  -0.80        4      328
      a half  -0.79        2      261
     strange  -0.79        1      221
         put  -0.76        7      404
     one and  -0.76        4      311
          it  -0.75      657    14583
        half  -0.75        9      455
       sense  -0.74        9      450
        been  -0.74       38     1198
        it's  -0.73      103     2699
        nice  -0.71        7      379
    annoying  -0.69        4      280
        done  -0.69        6      338
   it wasn't  -0.67        1      170
   finish it  -0.67        0      131
  i couldn't  -0.66        4      264
     friends  -0.66       10      436
      follow  -0.66        1      165
        fish  -0.66        1      165
  

In [13]:
print("\n15 random NOT-flagged reviews (first 300 characters):")
for text in others[TEXT_COL].sample(15, random_state=SEED): # a loop taking a random sample of 15 NOT-flagged reviews
    print("-", text[:300].replace("\n", " "), "\n")


15 random NOT-flagged reviews (first 300 characters):
- I typically love anything touched by Jon scieszka. I suppose I need to be an elementary-aged boy to get it? I don't get it at all. The book is totally annoying. 

- This is the lowest rating I've ever given to a book that I liked, but the problematic section with the slaves ruins the rest of the beautiful experience. It's gut-wrenching to behold. :( 

- Why in the world didn't they put a shirt on the little girl! Blech! Children in books should be clothed! 

- WHAT the what was this? Why did I waste my time???!!!   Could I please get those 5 minutes back? And the brain cells it killed, I need those, too, please.   Oy vey. 

- Yuck. Just yuck. This third book in the "Very first..." series which has previously covered Easter and Christmas doesn't make any improvement. I much prefer Ben Avery's work on the Twelve Apostles, the Torchlighter DVD on Perpetua, and I hear Simonetta Carr has a book on Irenaeus coming out as well. 

- I th

#### Topic Model of NOT-flagged reviews

In [14]:
NUMBER_OF_TOPICS = 8
tfidf = TfidfVectorizer(stop_words="english", min_df=10, max_df=0.5) # using TF-IDF to assign weights to words in the reviews
X = tfidf.fit_transform(others[TEXT_COL]) # transforming the NOT-flagged reviews into a matrix
nmf = NMF(n_components=NUMBER_OF_TOPICS, random_state=SEED, init="nndsvda", max_iter=400) # using NMF to find topics in the NOT-flagged reviews, with 8 topics, a random seed for reproducibility, and a maximum of 400 iterations
doc_topics = nmf.fit_transform(X) # transforming the matrix of NOT-flagged reviews into a matrix of topic weights for each review
terms = tfidf.get_feature_names_out() # getting the feature names (words) from the TF-IDF vectorizer

others = others.copy()
others["topic"] = doc_topics.argmax(axis=1) # assigning each review to the topic with the highest weight
for i, weights in enumerate(nmf.components_): # a loop that goes through each topic and prints the top 10 words for that topic, along with the number of reviews assigned to that topic
    top_words = [terms[j] for j in weights.argsort()[::-1][:10]]
    n_reviews = (others["topic"] == i).sum()
    print(f"\nTopic {i} ({n_reviews:,} reviews): {', '.join(top_words)}")


Topic 0 (2,364 reviews): children, little, child, way, good, time, people, old, things, boy

Topic 1 (968 reviews): didn, like, really, did, felt, care, enjoy, wanted, characters, feel

Topic 2 (972 reviews): read, hated, school, ve, grade, time, remember, reading, aloud, did

Topic 3 (1,170 reviews): story, illustrations, text, cute, nice, pictures, good, words, liked, stars

Topic 4 (1,003 reviews): just, couldn, finish, maybe, really, bad, wasn, funny, sorry, got

Topic 5 (423 reviews): boring, finish, plot, thought, really, long, ugh, good, finished, dull

Topic 6 (856 reviews): don, know, think, kids, like, understand, kid, want, maybe, remember

Topic 7 (821 reviews): books, series, reading, better, love, picture, ve, enjoy, good, kids
